# 怎样让本地语言模型参考资料回答问题？：参考答案

每道题先写预期结果再运行。练习中的 TODO 需要自行完成；能顺序运行不代表已完成练习。

In [1]:
from pathlib import Path
CASE_DIR = Path.cwd()
# 载入本例的可修改函数和数据，所有算法都在同目录 solution.py。
exec(compile((CASE_DIR / "solution.py").read_text(encoding="utf-8"), str(CASE_DIR / "solution.py"), "exec"))

验证集选定阈值 0.25000000000000006


`torch_dtype` is deprecated! Use `dtype` instead!


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


签到地点在哪里 一号楼大厅 [来源:D1]


下大雨活动转到哪里 体育馆 [来源:D3]
明天股票价格是多少 资料不足，无法回答。
校长的生日是什么时候 资料不足，无法回答。
{
  "answerable_retrieval_recall": 1.0,
  "phrase_support_proxy_rate": 1.0,
  "unanswerable_refusal_rate": 1.0,
  "false_refusal_rate": 0.0,
  "threshold": 0.25000000000000006,
  "test_questions": 4,
  "human_full_claim_review": "required; phrase proxy is not full verification",
  "model": "Qwen/Qwen3-0.6B",
  "elapsed_seconds": 12.849
}


## 1. 补全：核对引用
非拒答答案的引用必须指向实际检索来源。

In [2]:
for row in rows:
    if not row['abstained']:
        assert '[来源:' + row['retrieved_doc'] + ']' in row['answer_with_source']
assert len(rows) == 4
print('引用绑定通过；不是事实正确性证明')

引用绑定通过；不是事实正确性证明


## 2. 修改：提高拒答阈值
仅在验证集比较原阈值与0.8导致的拒答数。解释错误拒答可能增加。

In [3]:
original = sum((s < threshold for s in validation_scores))
strict = sum((s < 0.8 for s in validation_scores))
assert strict >= original
print('原阈值拒答', original, '严格阈值拒答', strict)

原阈值拒答 2 严格阈值拒答 4


## 3. 迁移：受约束的计算工具
只允许 add，参数必须为有限数值；拒绝未知工具和字符串参数，不执行代码字符串。

In [4]:
def call_tool(name, args):
    if name != 'add':
        raise ValueError('不允许的工具')
    if set(args) != {'a', 'b'}:
        raise ValueError('参数字段错误')
    if any((type(x) not in (int, float) or not np.isfinite(x) for x in args.values())):
        raise ValueError('数值无效')
    return args['a'] + args['b']
assert call_tool('add', {'a': 2, 'b': 3}) == 5
for name, args in [('shell', {}), ('add', {'a': '2', 'b': 3})]:
    try:
        call_tool(name, args)
    except ValueError:
        pass
    else:
        raise AssertionError('非法调用未拒绝')
print('工具边界检查通过')

工具边界检查通过
